In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cholesky
import os
import sys

sys.path.append(os.path.abspath(".."))  # Adds the parent directory to sys.path

from privateggm.model import optimize_lbfgs, mirror_descent
from privateggm.utils import *

# MaxEnt and Mirror Descent test (functions in model.py)

In [55]:
# generate a toy 3x3 covariance example
def generate_toy_example(Sigma_true, inds, noise_scale):
    """
    Generate a small 3x3 synthetic covariance matrix and parameters for testing.
    """

    # generate symmetric noise matrix
    lower_triangle_noise = np.tril(np.random.normal(0, noise_scale, Sigma_true.shape))
    tau = lower_triangle_noise + lower_triangle_noise.T - np.diag(np.diag(lower_triangle_noise))
    # perturb Sigma_true
    Sigma_tilde = Sigma_true + tau

    # select indices of the observed entries (only diagonal and lower triangle indices)
    rows, cols = inds[:, 0], inds[:, 1]
    Sigma_tilde_vec = Sigma_tilde[rows, cols]

    z_init = initialize_z(Sigma_tilde, inds)

    # For mirror descent
    Sigma_init = np.eye(Sigma_tilde.shape[0]) 

    return  Sigma_tilde, Sigma_tilde_vec, z_init, Sigma_init, tau

def test_lbfgs(Sigma_true, inds, noise_scale, max_iters=1000, verbose=False):
    """
    Run the L-BFGS optimization on the toy example.
    """
    Sigma_tilde, Sigma_tilde_vec, z_init, _, _ = generate_toy_example(Sigma_true, inds, noise_scale)
    
    d = Sigma_true.shape[0]  # Dimensionality
    
    # Run lbfgs optimization
    z_opt, g_vals, obj_vals = optimize_lbfgs(Sigma_tilde_vec, z_init, inds, d, max_iters=max_iters, verbose=verbose)
    
    # Convert optimized z into a precision matrix K
    K_z_opt = get_K(z_opt, inds, d)
    
    # Convert back to covariance matrix
    Sigma_opt = np.linalg.inv(K_z_opt)

    # Create boolean mask from indices
    mask = np.zeros((d, d), dtype=bool)
    for i, j in inds:
        mask[i, j] = True
        mask[j, i] = True  # Ensure symmetry

    # Compute errors
    mse_total = np.mean((Sigma_true - Sigma_opt)**2)
    obs_mse = np.mean((Sigma_tilde[mask] - Sigma_opt[mask])**2)
    true_obs_mse = np.mean((Sigma_true[mask] - Sigma_opt[mask])**2)
    
    # Print results
    print("\n" + "-"*50)
    print(f"True Covariance:\n{Sigma_true}")
    print(f"\nNoisy Covariance:\n{Sigma_tilde}")
    print(f"\nMaxEnt direct method result:\n{Sigma_opt}")
    print("\n" + "-"*50)
    print(f"MSE vs True (Total): {mse_total:.6f}")
    print(f"MSE vs True (Observed): {true_obs_mse:.6f}")
    print(f"MSE vs Noisy (Observed): {obs_mse:.6f}")
    print("-"*50 + "\n")

def test_mirror_descent(Sigma_true, inds, noise_scale, max_iters=20000, eta_init=0.2, tol=1e-8, verbose=True):

    # Generate toy dataset
    Sigma_tilde, _, _, Sigma_init, tau = generate_toy_example(Sigma_true, inds, noise_scale)

    d = Sigma_tilde.shape[0]

    # tau in case no noise
    if noise_scale == 0.0:
        tau = np.zeros_like(Sigma_true)
        tau[np.array(inds)[:,0], np.array(inds)[:,1]] = 1.0  # Set observed τ=1
    
    # Run Mirror Descent
    Sigma_opt, K_opt = mirror_descent(Sigma_init, Sigma_tilde, inds, tau, 
                                      max_iters=max_iters, eta_init=eta_init, tol=tol, verbose=verbose)


    # Create boolean mask from indices
    mask = np.zeros((d, d), dtype=bool)
    for i, j in inds:
        mask[i, j] = True
        mask[j, i] = True  # Ensure symmetry
   
    # Compute errors
    mse_total = np.mean((Sigma_true - Sigma_opt)**2)
    obs_mse = np.mean((Sigma_tilde[mask] - Sigma_opt[mask])**2)
    true_obs_mse = np.mean((Sigma_true[mask] - Sigma_opt[mask])**2)
    
    # Print results
    print("\n" + "-"*50)
    print(f"True Covariance:\n{Sigma_true}")
    print(f"\nNoisy Covariance:\n{Sigma_tilde}")
    print(f"\nMirror Descent result:\n{Sigma_opt}")
    print("\n" + "-"*50)
    print(f"MSE vs True (Total): {mse_total:.6f}")
    print(f"MSE vs True (Observed): {true_obs_mse:.6f}")
    print(f"MSE vs Noisy (Observed): {obs_mse:.6f}")
    print("-"*50 + "\n")

## 3 x 3 well-conditioned case

In [47]:
np.random.seed(42)

# Define a well-conditioned covariance matrix
d = 3
Sigma_true = np.array([[2.0, 0.5, 0.3],
                       [0.5, 1.5, 0.4],
                       [0.3, 0.4, 1.0]])

# choose inds
inds = np.array([[0,0], [1,1], [2,2], [1,0], [2,0]])

print("******** LBFGS TEST w/ noise ********\n")
test_lbfgs(Sigma_true, inds, noise_scale=0.2, max_iters=2000, verbose=True)

print("******** LBFGS TEST w/o noise (exact recovery) ********\n")
test_lbfgs(Sigma_true, inds, noise_scale=0.0, max_iters=2000, verbose=True)

print("\n******** MIRROR DESCENT TEST w/ noise ********\n")
test_mirror_descent(Sigma_true, inds, noise_scale=0.2, max_iters=5000, eta_init=0.01, tol=1e-6, verbose=True)

print("\n******** MIRROR DESCENT TEST w/o noise (exact recovery ********\n")
test_mirror_descent(Sigma_true, inds, noise_scale=0.0, max_iters=5000, eta_init=0.01, tol=1e-6, verbose=True)

******** LBFGS TEST w/ noise ********

Iteration 0: Loss = 4.016771e+00, Grad Sum = 2.840897e+00, Step Size = 1.000000e+00
Optimization converged at iteration 11: Loss Change = 4.8164233e-09

--------------------------------------------------
True Covariance:
[[2.  0.5 0.3]
 [0.5 1.5 0.4]
 [0.3 0.4 1. ]]

Noisy Covariance:
[[2.09934283 0.80460597 0.61584256]
 [0.80460597 1.45316933 0.55348695]
 [0.61584256 0.55348695 0.90610512]]

MaxEnt direct method result:
[[2.09934298 0.80460487 0.61584118]
 [0.80460487 1.4531698  0.23603042]
 [0.61584118 0.23603042 0.90610504]]

--------------------------------------------------
MSE vs True (Total): 0.051081
MSE vs True (Observed): 0.057994
MSE vs Noisy (Observed): 0.000000
--------------------------------------------------

******** LBFGS TEST w/o noise (exact recovery) ********

Iteration 0: Loss = 4.098612e+00, Grad Sum = 1.600000e+00, Step Size = 1.000000e+00
Optimization converged at iteration 10: Loss Change = 1.9589796e-10

----------------

# 5x5 matrix

In [53]:
np.random.seed(42)

d = 5
A = np.random.randn(d, d)
Sigma_true = A @ A.T + np.eye(d)  # ensure well-conditioned

noise_scale = 0.0

# choose inds
inds = np.array([[0,0], [1,1], [2,2], [3,3], [4,4], [1,0], [3,0], [2,1]])


print("******** LBFGS TEST w/ noise ********\n")
test_lbfgs(Sigma_true, inds, noise_scale=0.2, max_iters=2000, verbose=True)

print("******** LBFGS TEST w/o noise (exact recovery) ********\n")
test_lbfgs(Sigma_true, inds, noise_scale=0.0, max_iters=2000, verbose=True)

print("\n******** MIRROR DESCENT TEST w/ noise ********\n")
test_mirror_descent(Sigma_true, inds, noise_scale=0.2, max_iters=5000, eta_init=0.01, tol=1e-6, verbose=True)

print("\n******** MIRROR DESCENT TEST w/o noise (exact recovery ********\n")
test_mirror_descent(Sigma_true, inds, noise_scale=0.0, max_iters=5000, eta_init=0.01, tol=1e-7, verbose=True)

******** LBFGS TEST w/ noise ********

Iteration 0: Loss = 1.335913e+01, Grad Sum = 4.154868e+00, Step Size = 1.000000e+00
Optimization converged at iteration 9: Loss Change = 3.3800394e-09

--------------------------------------------------
True Covariance:
[[ 4.05979016 -0.67965499 -2.51916374 -0.98797549 -1.23950244]
 [-0.67965499  4.65246689 -0.47892836 -1.56662479 -0.27436542]
 [-2.51916374 -0.47892836  8.12618873  4.98172794  3.10724048]
 [-0.98797549 -1.56662479  4.98172794  5.259855    1.48831719]
 [-1.23950244 -0.27436542  3.10724048  1.48831719  5.52992126]]

Noisy Covariance:
[[ 4.08197467 -0.79999631 -2.76333247 -0.84028217 -1.38347128]
 [-0.79999631  5.02292253 -0.43715564 -1.53235113 -0.36649317]
 [-2.76333247 -0.43715564  7.7342547   4.95859828  3.31866492]
 [-0.84028217 -1.53235113  4.95859828  5.19963426  1.55704085]
 [-1.38347128 -0.36649317  3.31866492  1.55704085  5.17731322]]

MaxEnt direct method result:
[[ 4.08197684 -0.79999451  0.06962549 -0.8402903   0.       

# Pathological case (d=10)

In [58]:
np.random.seed(42)

# Load problematic Sigma_obs and set it as Sigma_true
Sigma_true = np.load("../pathologic_d10_sigma_tilde_dp.npy")

# Fetch the observed indices
inds = get_observed_Sigma_entries(Sigma_true)

# Noise scale needs to be 0 (we want to optimize the pathologic case directly)
noise_scale = 0.

print("******** LBFGS TEST ********\n")
test_lbfgs(Sigma_true, inds, noise_scale, max_iters=2000, verbose=True)

print("\n******** MIRROR DESCENT TEST ********\n")
test_mirror_descent(Sigma_true, inds, noise_scale, max_iters=2000, eta_init=0.01, tol=1e-6, verbose=True)

******** LBFGS TEST ********

Iteration 0: Loss = -1.600155e+01, Grad Sum = 1.579044e+00, Step Size = 1.000000e+00
Iteration 100: Loss = -9.764940e+02, Grad Sum = 1.812591e+07, Step Size = 1.000000e+00
Iteration 200: Loss = -1.228352e+03, Grad Sum = 3.663688e+07, Step Size = 1.000000e+00
Iteration 300: Loss = -1.495967e+03, Grad Sum = 9.773442e+07, Step Size = 1.000000e+00
Iteration 400: Loss = -1.778522e+03, Grad Sum = 2.649761e+08, Step Size = 1.000000e+00
Iteration 500: Loss = -2.073463e+03, Grad Sum = 5.525999e+07, Step Size = 1.000000e+00
Iteration 600: Loss = -2.354850e+03, Grad Sum = 3.483607e+08, Step Size = 1.000000e+00
Iteration 700: Loss = -2.640635e+03, Grad Sum = 7.043193e+08, Step Size = 1.000000e+00
Iteration 800: Loss = -2.932339e+03, Grad Sum = 4.115169e+08, Step Size = 1.000000e+00
Iteration 900: Loss = -3.207611e+03, Grad Sum = 3.189728e+08, Step Size = 1.000000e+00
Iteration 1000: Loss = -3.499929e+03, Grad Sum = 3.213323e+08, Step Size = 1.000000e+00
Iteration 1100

# ADMM attempt (DeepSeek)

In [59]:
def estimate_precision_matrix_admm(
        observed_covariance: np.ndarray, 
        observation_mask: np.ndarray, 
        entropy_weight: float, 
        penalty_param: float = 1.0, 
        max_iterations: int = 1000,
        convergence_tol: float = 1e-6,
        psd_jitter: float = 1e-8) -> np.ndarray:
    """
    ADMM solver for maximum entropy precision matrix estimation with explicit variable splitting.
    
    Splits the problem into two subproblems:
    1. Covariance estimation (data fidelity)
    2. Precision matrix estimation (entropy maximization + PSD constraints)
    
    Parameters:
        observed_covariance (np.ndarray): Noisy covariance matrix with NaNs for missing entries
        observation_mask (np.ndarray): Boolean mask indicating observed entries
        entropy_weight (float): Weight for entropy regularization (\u03bb in -\u03bb logdet term)
        penalty_param (float): Initial ADMM penalty parameter (\u03c1)
        max_iterations (int): Maximum number of ADMM iterations
        convergence_tol (float): Tolerance for primal/dual residual convergence
        psd_jitter (float): Small value to ensure positive definiteness
        
    Returns:
        np.ndarray: Estimated precision matrix \u0398 = \u03a3^{-1}
    """
    
    # Problem dimensions
    n = observed_covariance.shape[0]
    observed_entries = observation_mask.astype(bool)
    
    # ADMM variables with explicit roles
    covariance_estimate = np.eye(n, dtype=np.float64)  # Data fidelity estimate
    precision_estimate = np.eye(n, dtype=np.float64)   # Z: PSD/Entropy-constrained estimate
    dual_variable = np.zeros((n, n), dtype=np.float64)  # U: Lagrange multiplier
    
    # Track residuals for adaptive penalty adjustment
    primal_residual_history = []
    dual_residual_history = []
    
    # ADMM iteration
    for iteration in range(max_iterations):
        prev_precision = precision_estimate.copy()
        
        # --- Covariance Update (Data Fidelity) ---
        numerator = np.where(observed_entries, observed_covariance, 0.0) 
        numerator += penalty_param * (precision_estimate - dual_variable)
        denominator = np.where(observed_entries, 1.0, 0.0) + penalty_param
        covariance_estimate = numerator / denominator
        
        # Project to PSD cone
        eigenvalues, eigenvectors = np.linalg.eigh(covariance_estimate)
        eigenvalues = np.maximum(eigenvalues, psd_jitter)
        covariance_estimate = eigenvectors @ np.diag(eigenvalues) @ eigenvectors.T
        covariance_estimate = (covariance_estimate + covariance_estimate.T) / 2  

        # --- Precision Matrix Update (Entropy + PSD) ---
        constraint_matrix = covariance_estimate + dual_variable
        constraint_matrix = (constraint_matrix + constraint_matrix.T) / 2  
        
        eigenvalues, eigenvectors = np.linalg.eigh(constraint_matrix)
        adjusted_eigenvalues = (eigenvalues + np.sqrt(eigenvalues**2 + 4*entropy_weight/penalty_param)) / 2
        precision_estimate = eigenvectors @ np.diag(adjusted_eigenvalues) @ eigenvectors.T

        # --- Dual Variable Update (Constraint Enforcement) ---
        dual_variable += covariance_estimate - precision_estimate
        dual_variable = (dual_variable + dual_variable.T) / 2  

        # --- Convergence Check ---
        primal_residual = np.linalg.norm(covariance_estimate - precision_estimate, 'fro')
        dual_residual = penalty_param * np.linalg.norm(precision_estimate - prev_precision, 'fro')
        
        if primal_residual > 10 * dual_residual:
            penalty_param *= 2
            dual_variable /= 2  
        elif dual_residual > 10 * primal_residual:
            penalty_param /= 2
            dual_variable *= 2
            
        if primal_residual < convergence_tol and dual_residual < convergence_tol:
            break

    return np.linalg.inv(precision_estimate + psd_jitter*np.eye(n))

# Example Usage ----------------------------------------------------------------

n = 50
true_precision = np.eye(n) + 0.1*(np.random.rand(n, n) < 0.05).astype(float)
true_precision = (true_precision + true_precision.T)/2  
true_covariance = np.linalg.inv(true_precision)

observation_mask = np.random.rand(n, n) < 0.3
observation_mask = np.triu(observation_mask) | np.triu(observation_mask, 1).T  

observed_covariance = true_covariance.copy()
observed_covariance[observation_mask] += np.random.normal(0, 0.1, np.sum(observation_mask))
observed_covariance = np.where(observation_mask, observed_covariance, np.nan)

theta_est = estimate_precision_matrix_admm(
    observed_covariance=observed_covariance,
    observation_mask=observation_mask,
    entropy_weight=0.1,
    penalty_param=1.0,
    max_iterations=1000
)

assert np.all(np.linalg.eigvals(theta_est) > 0), "Precision matrix must be positive definite!"
print("ADMM converged to valid precision matrix")

ADMM converged to valid precision matrix


In [60]:
def test_admm():
    """
    Test the ADMM function on a small 3x3 covariance example.
    """
    # Define a well-conditioned covariance matrix
    d = 3
    Sigma_true = np.array([[2.0, 0.5, 0.3],
                           [0.5, 1.5, 0.4],
                           [0.3, 0.4, 1.0]])
    noise_scale = 0.1

    # Define observed indices mask
    inds = np.array([[0,0], [1,1], [2,2], [1,0], [2,0]])
    observation_mask = np.zeros((d, d), dtype=bool)
    for i, j in inds:
        observation_mask[i, j] = True
        observation_mask[j, i] = True  # Ensure symmetry

    # Create a copy of Sigma_true but with noise only on observed entries
    observed_covariance = Sigma_true.copy()
    
    # Add noise only to observed entries (without modifying NaN positions)
    noise_matrix = np.random.normal(0, noise_scale, (d, d))
    noise_matrix = (noise_matrix + noise_matrix.T) / 2  # Ensure symmetry
    observed_covariance[observation_mask] += noise_matrix[observation_mask]

    # Run ADMM solver
    entropy_weight = 0.1  # λ for entropy regularization
    penalty_param = 1.0   # ρ for ADMM penalty parameter

    theta_est = estimate_precision_matrix_admm(
        observed_covariance=observed_covariance,
        observation_mask=observation_mask,
        entropy_weight=entropy_weight,
        penalty_param=penalty_param,
        max_iterations=1000
    )

    # Compute errors
    mse_total = np.mean((Sigma_true - np.linalg.inv(theta_est)) ** 2)
    observed_errors = (Sigma_true[observation_mask] - np.linalg.inv(theta_est)[observation_mask]) ** 2
    mse_observed = np.mean(observed_errors)

    # Print results
    print(f"\nTrue Covariance Matrix Sigma:\n{Sigma_true}")
    print(f"\nNoisy Covariance Matrix Sigma:\n{observed_covariance}")
    print(f"\nEstimated Covariance Matrix (ADMM):\n{np.linalg.inv(theta_est)}")
    print(f"\nMSE (total) between true and estimated Sigma: {mse_total:.6f}")
    print(f"\nMSE (observed entries only) between true and estimated Sigma: {mse_observed:.6f}")


print("******** LBFGS TEST ********\n")
test_admm()

******** LBFGS TEST ********


True Covariance Matrix Sigma:
[[2.  0.5 0.3]
 [0.5 1.5 0.4]
 [0.3 0.4 1. ]]

Noisy Covariance Matrix Sigma:
[[2.10965471 0.44655602 0.37688149]
 [0.44655602 1.45402354 0.4       ]
 [0.37688149 0.4        1.09826831]]

Estimated Covariance Matrix (ADMM):
[[2.16120463 0.43262584 0.36201084]
 [0.43262584 1.52361247 0.07245755]
 [0.36201084 0.07245755 1.18704612]]

MSE (total) between true and estimated Sigma: 0.032541

MSE (observed entries only) between true and estimated Sigma: 0.011186


# Projected Gradient Descent + MaxEnt

In [64]:
from scipy.linalg import eigh
import cvxpy as cp

# Step A: PSD-projected GD
def projected_gradient_descent(S, eta=0.1, max_iters=1000, tol=1e-6):
    """
    Stage A: Projected Gradient Descent (PGD) to minimize ||Sigma - S||**2 over PSD matrices
    
    args:
        S: Observed noisy covariance matrix (may not be PSD)
        eta: Step size
        max_iters: Maximum number of iterations
        tol: Convergence tolerance
    
    Returns:
        Sigma_opt: PSD matrix close to S
    """
    d = S.shape[0]
    Sigma = np.eye(d)  # Sigma init
    prev_Sigma = Sigma.copy()
    
    for t in range(max_iters):
        # compute gradient 
        gradient = Sigma - S
        
        # gradient step
        Sigma_temp = Sigma - eta * gradient
        
        # project to PSD via eigenvalue thresholding
        eigvals, eigvecs = eigh(Sigma_temp, subset_by_index=[0, d-1])
        eigvals = np.maximum(eigvals, 0)  # threshold eigenvalues
        Sigma = eigvecs @ np.diag(eigvals) @ eigvecs.T
        
        # Check convergence
        if np.linalg.norm(Sigma - prev_Sigma, 'fro') < tol:
            break
        prev_Sigma = Sigma.copy()
    
    return Sigma

# step B: MaxEnt
def max_entropy_optimization(Sigma_psd, S, mask):
    """
    DEEPSEEK DRAFT using splitting cone solver cvxpy
    Need to compare with our lbfgs implementation.
    
    Stage B: Apply mask and maximize entropy on precision matrix K = Sigma^{-1}.
    
    Args:
        Sigma_sd: PSD matrix from step A
        S: observed (noisy) matrix
        mask: boolean matrix indicating observed entries
    
    Returns:
        Sigma_refined: Final PSD matrix with observed entries fixed
    """
    n = Sigma_psd.shape[0]
    
    # Apply mask: Overwrite observed entries with original S
    Sigma_masked = Sigma_psd.copy()
    Sigma_masked[mask] = S[mask]
    
    # Define optimization variables and constraints
    K = cp.Variable((n, n), symmetric=True)
    constraints = [
        K >> 0,  # K must be positive definite
        cp.matmul(K, Sigma_masked) == np.eye(n)  # Enforce K^{-1} matches masked Sigma
    ]
    
    # Objective: max log(det(K)) - tr(KSigma_masked)
    objective = cp.Maximize(cp.log_det(K) - cp.trace(K @ Sigma_masked))
    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.SCS)
    
    if problem.status != 'optimal':
        raise ValueError("Max entropy optimization failed")
    
    # Return refined covariance matrix
    K_opt = K.value
    Sigma_refined = np.linalg.inv(K_opt)
    return Sigma_refined


def test_pgd_maxent(Sigma_true, inds, noise_scale, max_iters=1000, eta=0.1, tol=1e-6, verbose=True):
    """
    Test the PGD + Max Entropy pipeline.
    
    Args:
        Sigma_true: Ground truth covariance matrix
        inds: Array of observed indices [[i,j], ...]
        noise_scale: Standard deviation of additive noise
        max_iters: Max iterations for PGD
        eta: Step size for PGD
        tol: Convergence tolerance for PGD
        verbose: Print progress
    """
    # Generate noisy data (Sigma_tilde is fully observed but noisy)
    Sigma_tilde, _, _, _, _ = generate_toy_example(Sigma_true, inds, noise_scale)
    n = Sigma_true.shape[0]
    
    # Create boolean mask from indices
    mask = np.zeros((n, n), dtype=bool)
    for i, j in inds:
        mask[i, j] = True
        mask[j, i] = True  # Ensure symmetry
    
    # Stage A: Projected Gradient Descent
    if verbose:
        print("\nRunning Projected Gradient Descent...")
    Sigma_psd = projected_gradient_descent(Sigma_tilde, eta=eta, max_iters=max_iters, tol=tol)
    
    # Stage B: Max Entropy Optimization
    if verbose:
        print("\nRunning Max Entropy Refinement...")
    try:
        Sigma_refined = max_entropy_optimization(Sigma_psd, Sigma_tilde, mask)
    except Exception as e:
        print(f"Max Entropy failed: {str(e)}")
        return
    
    # Compute errors
    mse_total = np.mean((Sigma_true - Sigma_refined)**2)
    obs_mse = np.mean((Sigma_tilde[mask] - Sigma_refined[mask])**2)
    true_obs_mse = np.mean((Sigma_true[mask] - Sigma_refined[mask])**2)
    
    # Print results
    print("\n" + "-"*50)
    print(f"True Covariance:\n{Sigma_true.round(5)}")
    print(f"\nNoisy Covariance:\n{Sigma_tilde.round(5)}")
    print(f"\nPGD + MaxEnt Result:\n{Sigma_refined.round(5)}")
    print("\n" + "-"*50)
    print(f"MSE vs True (Total): {mse_total:.6f}")
    print(f"MSE vs True (Observed): {true_obs_mse:.6f}")
    print(f"MSE vs Noisy (Observed): {obs_mse:.6f}")
    print("-"*50 + "\n")

## 3 x 3 well-conditioned case

In [66]:
np.random.seed(42)

# Define a well-conditioned covariance matrix
d = 3
Sigma_true = np.array([[2.0, 0.5, 0.3],
                       [0.5, 1.5, 0.4],
                       [0.3, 0.4, 1.0]])

# choose inds
inds = np.array([[0,0], [1,1], [2,2], [1,0], [2,0]])

# Create BOOLEAN mask (not float)
mask = np.zeros_like(S, dtype=bool)  # Fix: dtype=bool
for i, j in inds:
    mask[i, j] = True
    mask[j, i] = True  # ensure symmetry

# test with noise
test_pgd_maxent(Sigma_true, inds, noise_scale=0.1)

# test without noise (exact recovery)
test_pgd_maxent(Sigma_true, inds, noise_scale=0.0)


Running Projected Gradient Descent...

Running Max Entropy Refinement...

--------------------------------------------------
True Covariance:
[[2.  0.5 0.3]
 [0.5 1.5 0.4]
 [0.3 0.4 1. ]]

Noisy Covariance:
[[2.04967 0.6523  0.45792]
 [0.6523  1.47658 0.47674]
 [0.45792 0.47674 0.95305]]

PGD + MaxEnt Result:
[[2.04967 0.6523  0.45792]
 [0.6523  1.47658 0.47674]
 [0.45792 0.47674 0.95305]]

--------------------------------------------------
MSE vs True (Total): 0.012585
MSE vs True (Observed): 0.014499
MSE vs Noisy (Observed): 0.000000
--------------------------------------------------


Running Projected Gradient Descent...

Running Max Entropy Refinement...

--------------------------------------------------
True Covariance:
[[2.  0.5 0.3]
 [0.5 1.5 0.4]
 [0.3 0.4 1. ]]

Noisy Covariance:
[[2.  0.5 0.3]
 [0.5 1.5 0.4]
 [0.3 0.4 1. ]]

PGD + MaxEnt Result:
[[2.  0.5 0.3]
 [0.5 1.5 0.4]
 [0.3 0.4 1. ]]

--------------------------------------------------
MSE vs True (Total): 0.000000
M

## 5 x 5 well-conditioned case

In [63]:
np.random.seed(42)

d = 5
A = np.random.randn(d, d)
Sigma_true = A @ A.T + np.eye(d)  # ensure well-conditioned

# choose inds
inds = np.array([[0,0], [1,1], [2,2], [3,3], [4,4], [1,0], [4,0], [2,1]])

# Create BOOLEAN mask (not float)
mask = np.zeros_like(S, dtype=bool)  # Fix: dtype=bool
for i, j in inds:
    mask[i, j] = True
    mask[j, i] = True  # ensure symmetry

# test with noise
test_pgd_maxent(Sigma_true, inds, noise_scale=0.1)

# test without noise (exact recovery)
test_pgd_maxent(Sigma_true, inds, noise_scale=0.0)


Running Projected Gradient Descent...

Running Max Entropy Refinement...

--------------------------------------------------
True Covariance:
[[ 4.05979 -0.67965 -2.51916 -0.98798 -1.2395 ]
 [-0.67965  4.65247 -0.47893 -1.56662 -0.27437]
 [-2.51916 -0.47893  8.12619  4.98173  3.10724]
 [-0.98798 -1.56662  4.98173  5.25985  1.48832]
 [-1.2395  -0.27437  3.10724  1.48832  5.52992]]

Noisy Covariance:
[[ 4.07088 -0.73983 -2.64125 -0.91413 -1.31149]
 [-0.73983  4.83769 -0.45804 -1.54949 -0.32043]
 [-2.64125 -0.45804  7.93022  4.97016  3.21295]
 [-0.91413 -1.54949  4.97016  5.22974  1.52268]
 [-1.31149 -0.32043  3.21295  1.52268  5.35362]]

PGD + MaxEnt Result:
[[ 4.07088 -0.73983 -2.64125 -0.91413 -1.31149]
 [-0.73983  4.83769 -0.45804 -1.54949 -0.32043]
 [-2.64125 -0.45804  7.93022  4.97016  3.21295]
 [-0.91413 -1.54949  4.97016  5.22974  1.52268]
 [-1.31149 -0.32043  3.21295  1.52268  5.35362]]

--------------------------------------------------
MSE vs True (Total): 0.007753
MSE vs True